In [17]:
import sys, os
sys.path.append(os.path.abspath('..'))
import time
import joblib
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from src.preprocessing import  get_preprocessor
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

In [2]:
sys.path.append(os.getcwd())

from src.app import app

In [3]:
client = TestClient(app)
model = joblib.load("../models/xgboost_pipeline.joblib")
cleaned_data = joblib.load("../data/cleaned_telco_data.joblib")

In [4]:
X = cleaned_data.drop("Churn", axis=1)
y = LabelEncoder().fit_transform(cleaned_data["Churn"])

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [6]:
api_features = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "tenure",
    "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
    "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod",
    "MonthlyCharges", "TotalCharges"
]

In [7]:
sample_row = X_test[api_features].iloc[0]
valid_payload = sample_row.to_dict()


print("Setup complete. Valid payload ready!")

Setup complete. Valid payload ready!


In [8]:
res_root = client.get("/")
assert res_root.status_code == 200
assert "status" in res_root.json()
print("1. Root endpoint test passed:", res_root.json())


1. Root endpoint test passed: {'status': 'API is running', 'model': 'XGBoost Churn Pipeline', 'docs_url': '/docs'}


In [9]:
res_info = client.get("/model-info")
info_data = res_info.json()
assert res_info.status_code == 200
assert info_data["model_version"] == "1.0.0"
assert info_data["number_of_features"] == X.shape[1]
assert len(info_data["classes"]) == 2
print("2. Model-info & feature contract test passed:", info_data)

2. Model-info & feature contract test passed: {'model_name': 'telco_churn_classifier', 'model_version': '1.0.0', 'model_type': 'XGBClassifier (Pipeline)', 'number_of_features': 20, 'classes': ['no', 'yes']}


In [10]:
res_pred = client.post("/predict", json=valid_payload)
assert res_pred.status_code == 200

prediction_result = res_pred.json()
print("Prediction Result:", prediction_result)

# Validate response keys, class names, and probability range
assert isinstance(prediction_result["prediction"], int)
assert prediction_result["prediction"] in [0, 1]
assert prediction_result["class_name"] in ["No", "Yes"]
assert isinstance(prediction_result["probability"], float)
assert 0.0 <= prediction_result["probability"] <= 1.0

print("3. Valid prediction, class, and probability range tests passed!")

Prediction Result: {'prediction': 0, 'class_name': 'No', 'probability': 0.9753}
3. Valid prediction, class, and probability range tests passed!


In [11]:
# 1. Missing required field
missing_payload = valid_payload.copy()
missing_payload.pop("tenure")

# 2. Wrong data type
wrong_type_payload = valid_payload.copy()
wrong_type_payload["MonthlyCharges"] = "not-a-number"

# 3. Unexpected extra field
extra_field_payload = valid_payload.copy()
extra_field_payload["unexpected_feature"] = 999

validation_matrix = [
    ("Valid Payload", valid_payload, 200),
    ("Missing Field", missing_payload, 422),
    ("Wrong Data Type", wrong_type_payload, 422),
    ("Extra Field", extra_field_payload, 422),
]




In [12]:
for test_name, payload, expected_status in validation_matrix:
    resp = client.post("/predict", json=payload)
    assert resp.status_code == expected_status, f"Failed on {test_name}"
    print(f" - {test_name}: Passed (Status {resp.status_code})")

print("4. Validation matrix passed!")

 - Valid Payload: Passed (Status 200)
 - Missing Field: Passed (Status 422)
 - Wrong Data Type: Passed (Status 422)
 - Extra Field: Passed (Status 422)
4. Validation matrix passed!


In [14]:
sample_df = sample_row.to_frame().T


direct_pred = int(model.predict(sample_df).item())
direct_prob = float(model.predict_proba(sample_df).flatten()[direct_pred])


api_res = client.post("/predict", json=valid_payload).json()
api_pred = api_res["prediction"]
api_prob = api_res["probability"]

print(f"Direct Model -> Class: {direct_pred}, Prob: {round(direct_prob, 4)}")
print(f"FastAPI      -> Class: {api_pred}, Prob: {api_prob}")

assert direct_pred == api_pred
assert np.isclose(api_prob, round(direct_prob, 4), rtol=1e-4, atol=1e-4)

print("5. API vs Direct Model consistency passed!")

Direct Model -> Class: 0, Prob: 0.9753
FastAPI      -> Class: 0, Prob: 0.9753
5. API vs Direct Model consistency passed!


In [18]:

repeated_preds = [
    client.post("/predict", json=valid_payload).json()["prediction"]
    for _ in range(5)
]
assert len(set(repeated_preds)) == 1
print("6. Repeated prediction stability passed:", repeated_preds)


for idx in range(5):
    row_payload = X_test[api_features].iloc[idx].to_dict()
    r = client.post("/predict", json=row_payload)
    assert r.status_code == 200
    assert r.json()["prediction"] in [0, 1]
    assert 0.0 <= r.json()["probability"] <= 1.0
print("7. Multiple-sample API test (5 customers) passed!")


start = time.perf_counter()
_ = client.post("/predict", json=valid_payload)
latency_ms = (time.perf_counter() - start) * 1000
print(f"8. Latency check passed: {latency_ms:.2f} ms")


test_probs = model.predict_proba(X_test)[:, 1]
test_roc_auc = roc_auc_score(y_test, test_probs)
MIN_ACCEPTABLE_ROC_AUC = 0.80

assert test_roc_auc >= MIN_ACCEPTABLE_ROC_AUC
print(f"9. Model Quality Gate passed! (ROC-AUC: {test_roc_auc:.4f} >= {MIN_ACCEPTABLE_ROC_AUC})")


6. Repeated prediction stability passed: [0, 0, 0, 0, 0]
7. Multiple-sample API test (5 customers) passed!
8. Latency check passed: 44.59 ms
9. Model Quality Gate passed! (ROC-AUC: 0.8437 >= 0.8)
